In [ ]:
%pip install langchain-openai python-dotenv "pydantic>=2,<3"

In [1]:
from pydantic import BaseModel, Field

class OrderItem(BaseModel):
    product: str
    quantity: int = Field(ge = 1, le = 20, strict = True, description = "Quantity must be between 1 and 20")

In [4]:
item = OrderItem(product="cold_coffee", quantity=5)  # Valid
print(item.product)
print(item.quantity)

cold_coffee
5


In [5]:
import os
from pathlib import Path
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

load_dotenv()
llm = ChatOpenAI(
    model="google/gemini-2.5-flash",
    api_key=os.environ["OPEN_ROUTER_API_KEY"],
    base_url="https://openrouter.ai/api/v1",
    temperature=0,
)

In [10]:
## How do I tell LLM that you need to give me output in this structure?
item_llm = llm.with_structured_output(OrderItem)
item = item_llm.invoke("I want to order 5 cold coffees")
print(item.product)
print(item.quantity)

cold coffees
5


/Users/sameerdubey/miniconda3/envs/ai-dev/lib/python3.12/site-packages/pydantic/main.py:463: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [input_value=OrderItem(product='cold coffees', quantity=5), input_type=OrderItem])
  return self.__pydantic_serializer__.to_python(


In [8]:
llm.invoke("I want to order 5 cold coffees")

AIMessage(content='Sure, I can help you with that. What kind of cold coffees would you like? Do you have any specific preferences, like:\n\n*   **Type of coffee:** Iced latte, iced americano, cold brew, frappuccino, etc.?\n*   **Size:** Small, medium, large?\n*   **Milk preference:** Dairy (whole, skim, etc.) or non-dairy (almond, oat, soy)?\n*   **Sweetener:** Sugar, syrup (vanilla, caramel), artificial sweetener, or unsweetened?\n*   **Any other additions:** Whipped cream, extra shot, flavorings?\n\nOnce I have these details, I can place the order for you!', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 146, 'prompt_tokens': 8, 'total_tokens': 154, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': None, 'image_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': 0, 'cached_tokens': 0, 'video_tokens': 0}, 'cost': 0.0003674, '

In [9]:
item_llm.invoke("Hi How are you?")

/Users/sameerdubey/miniconda3/envs/ai-dev/lib/python3.12/site-packages/pydantic/main.py:463: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [input_value=OrderItem(product='unknown', quantity=1), input_type=OrderItem])
  return self.__pydantic_serializer__.to_python(


OrderItem(product='unknown', quantity=1)

In [11]:
from pydantic import ValidationError

try:
    OrderItem(product="cold coffee", quantity=-2)
except ValidationError as error:
    print(error)

1 validation error for OrderItem
quantity
  Input should be greater than or equal to 1 [type=greater_than_equal, input_value=-2, input_type=int]
    For further information visit https://errors.pydantic.dev/2.11/v/greater_than_equal


In [ ]:
from typing import Literal

class MenuItem(BaseModel):
    product_id: Literal[
        "cold_coffee", "paneer_tikka", "veg_sandwich", "lemonade"
    ]
    quantity: int = Field(ge=1, le=20, strict=True)

    #{
    }

In [13]:
menu_llm = llm.with_structured_output(MenuItem, method="function_calling")

item = menu_llm.invoke(
    "Extract the menu item and quantity from the following order:"
    "Customer: Two cold coffees, please and one paneer tikka please"
)
print(item.model_dump_json(indent=2))

{
  "product_id": "cold_coffee",
  "quantity": 2
}


In [14]:
class OrderSelection(BaseModel):
    items: list[MenuItem] = Field(min_length=1, max_length=4)

In [15]:
order_llm = llm.with_structured_output(
    OrderSelection,
    method="function_calling",
)

instructions = """Extract the customer's order using the menu product IDs.
Menu: Cold Coffee, Paneer Tikka, Veg Sandwich, Lemonade.
Use the quantities explicitly requested. Do not include prices or payment details.
"""

order = order_llm.invoke([
    ("system", instructions),
    ("human", "Two cold coffees and one paneer tikka, please."),
])
print(order.model_dump_json(indent=2))

{
  "items": [
    {
      "product_id": "cold_coffee",
      "quantity": 2
    },
    {
      "product_id": "paneer_tikka",
      "quantity": 1
    }
  ]
}
